In [0]:
import dlt
from pyspark.sql.functions import col

# --- SILVER LAYER: FACTS & QUARANTINE ---

# 1. Bookings Fact
booking_rules = {
    "valid_booking_id": "booking_id IS NOT NULL",
    "valid_dates": "check_out > check_in",
    "valid_property": "property_id IS NOT NULL"
}

@dlt.table(name="silver.silver_bookings", comment="Clean and deduplicated bookings fact.")
@dlt.expect_all_or_drop(booking_rules)
def silver_bookings():
    return (
        dlt.read_stream("bronze.bronze_bookings")
        .dropDuplicates(["booking_id"])
        .withColumn("check_in", col("check_in").cast("date"))
        .withColumn("check_out", col("check_out").cast("date"))
    )

@dlt.table(name="ops.quarantine_bookings", comment="Quarantined bookings.")
def quarantine_bookings():
    return (
        dlt.read_stream("bronze.bronze_bookings")
        .filter("NOT (booking_id IS NOT NULL AND check_out > check_in AND property_id IS NOT NULL)")
    )

# 2. Payments Fact
payment_rules = {
    "valid_payment_id": "payment_id IS NOT NULL",
    "valid_booking_id": "booking_id IS NOT NULL",
    "valid_amount": "amount >= 0"
}

@dlt.table(name="silver.silver_payments", comment="Clean and deduplicated payments fact.")
@dlt.expect_all_or_drop(payment_rules)
def silver_payments():
    return (
        dlt.read_stream("bronze.bronze_payments")
        .dropDuplicates(["payment_id"])
        .withColumn("amount", col("amount").cast("decimal(10,2)"))
    )

@dlt.table(name="ops.quarantine_payments", comment="Quarantined payments.")
def quarantine_payments():
    return (
        dlt.read_stream("bronze.bronze_payments")
        .filter("NOT (payment_id IS NOT NULL AND booking_id IS NOT NULL AND amount >= 0)")
    )

# --- SILVER LAYER: SCD TYPE 2 DIMENSION ---

dlt.create_streaming_table(
    name="silver.silver_properties",
    comment="SCD Type 2 properties dimension tracking base price changes."
)

dlt.apply_changes(
    target="silver.silver_properties",
    source="bronze.bronze_properties",
    keys=["property_id"],
    sequence_by=col("ingestion_time"),
    except_column_list=["source_file_path", "batch_number", "ingestion_time"],
    stored_as_scd_type=2
)

# --- SILVER LAYER: STANDARD DIMENSIONS ---

def generate_silver_dim(entity_name, key_col):
    @dlt.table(name=f"silver.silver_{entity_name}", comment=f"Clean {entity_name} dimension.")
    def create_dim_table():
        return dlt.read_stream(f"bronze.bronze_{entity_name}").dropDuplicates([key_col])

generate_silver_dim("destinations", "destination_id")
generate_silver_dim("users", "user_id")
generate_silver_dim("hosts", "host_id")